### RAG Pipelines - Data Ingestion to Vector DB Pipeline

In [1]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

C:\Users\Kamran\AppData\Local\Temp\ipykernel_17888\3933654057.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
d:\Langchain\agentic_langgraph\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def process_all_pdfs(pdf_directory):
    """Process all PDF files in the specified directory and return a list of documents with metadata."""
    all_documents = []
    pdf_dir = Path(pdf_directory)

    # Find all PDF files in the directory
    pdf_files = list(pdf_dir.glob("**/*.pdf"))

    print(f"Found {len(pdf_files)} PDF files in {pdf_directory}")

    for pdf_file in pdf_files:
        print(f"\nProcessing {pdf_file.name}...")
        try:
            # Load the PDF file
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()

            # Add source information to metadata
            for doc in documents:
                doc.metadata["source_file"] = pdf_file.name  # Add the source file name to metadata
                doc.metadata["file_type"] = 'pdf'  # Add the file type to metadata

            all_documents.extend(documents)
            print(f"Loaded {len(documents)} pages.")
            
        except Exception as e:
            print(f"Error processing {pdf_file}: {e}")

    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

In [3]:
# Process all PDFs in the specified directory
all_pdf_documents = process_all_pdfs("../data/pdf_files/")

Found 2 PDF files in ../data/pdf_files/

Processing ai.pdf...
Loaded 9 pages.

Processing quantum.pdf...
Loaded 10 pages.

Total documents loaded: 19


In [4]:
all_pdf_documents

[Document(metadata={'producer': 'WeasyPrint 69.0', 'creator': 'PyPDF', 'creationdate': '', 'source': '..\\data\\pdf_files\\ai.pdf', 'total_pages': 9, 'page': 0, 'page_label': '1', 'source_file': 'ai.pdf', 'file_type': 'pdf'}, page_content='The AI Revolution: Unlocking\nBusiness Value and Strategic\nAdvantage\nA  Comprehensive  Analysis  of  Artificial  Intelligence  Trends,  Opportunities,  and\nChallenges\nPREPARED BY:  InsightSwarm Intelligence Agent\nDATE OF ISSUE:  August 2026'),
 Document(metadata={'producer': 'WeasyPrint 69.0', 'creator': 'PyPDF', 'creationdate': '', 'source': '..\\data\\pdf_files\\ai.pdf', 'total_pages': 9, 'page': 1, 'page_label': '2', 'source_file': 'ai.pdf', 'file_type': 'pdf'}, page_content='Table of Contents\nMARKET SIZE\n145.2B\n+12.4% CAGR\n(2020-2026)\nADOPTION RATE\n78%\nAcross Fortune 500\nCompanies\nFUNDING LEVEL\n18.4B\nTotal Venture Capital\nInflow\nENTERPRISE USERS\n4.2M\nActive Deployments\nglobally\n...............................................

In [5]:
def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    """Split documents into smaller chunks."""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks.")

    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")  # Print first 500 characters
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

In [6]:
chunks = split_documents(all_pdf_documents)
chunks

Split 19 documents into 32 chunks.

Example chunk:
Content: The AI Revolution: Unlocking
Business Value and Strategic
Advantage
A  Comprehensive  Analysis  of  Artificial  Intelligence  Trends,  Opportunities,  and
Challenges
PREPARED BY:  InsightSwarm Intelli...
Metadata: {'producer': 'WeasyPrint 69.0', 'creator': 'PyPDF', 'creationdate': '', 'source': '..\\data\\pdf_files\\ai.pdf', 'total_pages': 9, 'page': 0, 'page_label': '1', 'source_file': 'ai.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'WeasyPrint 69.0', 'creator': 'PyPDF', 'creationdate': '', 'source': '..\\data\\pdf_files\\ai.pdf', 'total_pages': 9, 'page': 0, 'page_label': '1', 'source_file': 'ai.pdf', 'file_type': 'pdf'}, page_content='The AI Revolution: Unlocking\nBusiness Value and Strategic\nAdvantage\nA  Comprehensive  Analysis  of  Artificial  Intelligence  Trends,  Opportunities,  and\nChallenges\nPREPARED BY:  InsightSwarm Intelligence Agent\nDATE OF ISSUE:  August 2026'),
 Document(metadata={'producer': 'WeasyPrint 69.0', 'creator': 'PyPDF', 'creationdate': '', 'source': '..\\data\\pdf_files\\ai.pdf', 'total_pages': 9, 'page': 1, 'page_label': '2', 'source_file': 'ai.pdf', 'file_type': 'pdf'}, page_content='Table of Contents\nMARKET SIZE\n145.2B\n+12.4% CAGR\n(2020-2026)\nADOPTION RATE\n78%\nAcross Fortune 500\nCompanies\nFUNDING LEVEL\n18.4B\nTotal Venture Capital\nInflow\nENTERPRISE USERS\n4.2M\nActive Deployments\nglobally\n...............................................

### Embedding and Vector Store

In [7]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [8]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the EmbeddingManager

        Args:
            model_name : HuggingFace model name for sentence embeddings 
        """

        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model."""
        try:
            print(f"Loading embedding model: {self.model_name}...")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model Loaded successfully. Embedding dimension: {self.model.get_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading embedding model {self.model_name}: {e}")

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts.

        Args:
            texts : List of strings to embed
        
        Returns:
            np.ndarray : Array of embeddings
        """

        if not self.model:
            raise ValueError("Embedding model is not loaded.")
        
        try:
            print(f"Generating embeddings for {len(texts)} texts...")
            embeddings = self.model.encode(texts, show_progress_bar=True, convert_to_numpy=True)
            embeddings = np.asarray(embeddings)
            print(f"Generated embeddings with shape: {embeddings.shape}")
            return embeddings
        except Exception as e:
            print(f"Error generating embeddings: {e}")
            return np.array([])

In [9]:
# Initialize the EmbeddingManager
embedding_manager = EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2455.12it/s]


Model Loaded successfully. Embedding dimension: 384


### Vector Store

In [14]:
class VectorStore:
    """Manages a vector store for document embeddings using ChromaDB"""

    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the VectorStore

        Args:
            collection_name : Name of the ChromaDB collection
            persist_directory : Directory to persist the vector store
        """

        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize the ChromaDB client and collection."""
        try:
            # Create persistence ChromaDB client
            print(f"Initializing ChromaDB client...")
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)

            # Get or create the collection
            self.collection = self.client.get_or_create_collection(
                name = self.collection_name,
                metadata = {"description": "Collection for PDF document embeddings"}
                )
            print(f"ChromaDB collection '{self.collection_name}' initialized successfully.")
            print(f"Existing documents in collection: {self.collection.count()}")
        except Exception as e:
            print(f"Error initializing ChromaDB: {e}")

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store.

        Args:
            documents : List of document metadata dictionaries
            embeddings : Corresponding embeddings as a numpy array
        """

        if not self.collection:
            raise ValueError("ChromaDB collection is not initialized.")

        if len(documents) != embeddings.shape[0]:
            raise ValueError("Number of documents and embeddings must match.")

        # Prepare data for insertion
        ids = []
        metadatas = []
        documents_texts = []
        embeddings_list = [] 

        print(f"Adding {len(documents)} documents to the vector store...")

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate a unique ID for each document
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            # Prepare metadata
            metadata = dict(doc.metadata)  # Copy metadata
            metadata['doc_index'] = i  # Add index to metadata
            metadata['content_length'] = len(doc.page_content)  # Add content length to metadata
            metadatas.append(metadata)

            # Document Content
            documents_texts.append(doc.page_content)

            # Embeddings
            embeddings_list.append(embedding.tolist())  # Convert numpy array to list

        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                metadatas=metadatas,
                documents=documents_texts,
                embeddings=embeddings_list
            )
            print(f"Successfully added {len(documents)} documents to the vector store.")
            print(f"Total documents in collection after addition: {self.collection.count()}")
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")

In [11]:
vector_store = VectorStore()
vector_store

Initializing ChromaDB client...
ChromaDB collection 'pdf_documents' initialized successfully.
Existing documents in collection: 0


In [12]:
chunks

[Document(metadata={'producer': 'WeasyPrint 69.0', 'creator': 'PyPDF', 'creationdate': '', 'source': '..\\data\\pdf_files\\ai.pdf', 'total_pages': 9, 'page': 0, 'page_label': '1', 'source_file': 'ai.pdf', 'file_type': 'pdf'}, page_content='The AI Revolution: Unlocking\nBusiness Value and Strategic\nAdvantage\nA  Comprehensive  Analysis  of  Artificial  Intelligence  Trends,  Opportunities,  and\nChallenges\nPREPARED BY:  InsightSwarm Intelligence Agent\nDATE OF ISSUE:  August 2026'),
 Document(metadata={'producer': 'WeasyPrint 69.0', 'creator': 'PyPDF', 'creationdate': '', 'source': '..\\data\\pdf_files\\ai.pdf', 'total_pages': 9, 'page': 1, 'page_label': '2', 'source_file': 'ai.pdf', 'file_type': 'pdf'}, page_content='Table of Contents\nMARKET SIZE\n145.2B\n+12.4% CAGR\n(2020-2026)\nADOPTION RATE\n78%\nAcross Fortune 500\nCompanies\nFUNDING LEVEL\n18.4B\nTotal Venture Capital\nInflow\nENTERPRISE USERS\n4.2M\nActive Deployments\nglobally\n...............................................

In [13]:
# Convert text to embeddings
texts = [doc.page_content for doc in chunks]
texts

['The AI Revolution: Unlocking\nBusiness Value and Strategic\nAdvantage\nA  Comprehensive  Analysis  of  Artificial  Intelligence  Trends,  Opportunities,  and\nChallenges\nPREPARED BY:  InsightSwarm Intelligence Agent\nDATE OF ISSUE:  August 2026',
 'Table of Contents\nMARKET SIZE\n145.2B\n+12.4% CAGR\n(2020-2026)\nADOPTION RATE\n78%\nAcross Fortune 500\nCompanies\nFUNDING LEVEL\n18.4B\nTotal Venture Capital\nInflow\nENTERPRISE USERS\n4.2M\nActive Deployments\nglobally\n...............................................................................................................................................................\n...........................................................1. Introduction & Context\n...............................................................................................................................................................\n...........................................................2. Market Landscape & Analysis\n.........................

In [15]:
# Generate embeddings for the chunks
embeddings = embedding_manager.generate_embeddings(texts)

# Store in the vector store
vector_store.add_documents(chunks, embeddings)

Generating embeddings for 32 texts...


Batches: 100%|██████████| 1/1 [00:01<00:00,  1.92s/it]

Generated embeddings with shape: (32, 384)
Adding 32 documents to the vector store...
Successfully added 32 documents to the vector store.
Total documents in collection after addition: 32
